<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="../../../images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    3.lab.1. Wisconsin Cancer Classification - Neural Networks
  </strong>
</div>

### Background

You will continue working with the Wisconsin Diagnostic Cancer Dataset, this time implementing neural networks using PyTorch.

The dataset includes:
- 569 instances
- 30 numeric features computed from the cell nuclei present in the image
- Binary classification: **Malignant** or **Benign** diagnosis

### Description

Your task is to implement neural networks using PyTorch to:

- Design and implement a multilayer neural network
- Experiment with different network architectures
- Compare performance with previous logistic regression results
- Visualize the training process and decision boundaries

### Deliverables

- A **ipynb** (Jupyter Notebook) file called **wisconsin-cancer-classification-nn.ipynb**
- The notebook should include visualizations of training and performance metrics
- A comparison with logistic regression results from week 2
- Use the provided template as a starting point

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 1. Carregar e Explorar o Dataset

In [ ]:
raw_data = load_breast_cancer(as_frame=True)
df = raw_data.data.copy()
df['target'] = raw_data.target
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['diagnosis'].value_counts())
print(f'\nFeature names ({len(raw_data.feature_names)}):')
print(raw_data.feature_names)
df.head()

# 2. Pré-processamento dos Dados

In [ ]:
X = raw_data.data.values
y = raw_data.target.values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set size: {len(X_train)}')
print(f'Test set size: {len(X_test)}')
print(f'\nX_train shape: {X_train_scaled.shape}')
print(f'X_test shape: {X_test_scaled.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

In [ ]:
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 3. Baseline de Regressão Logística

In [ ]:
lr = LogisticRegression(max_iter=10000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print(f'Logistic Regression Baseline')
print(f'Accuracy: {accuracy_lr:.4f}')
print(f'\nClassification Report:')
print(classification_report(y_test, y_pred_lr, target_names=['Malignant', 'Benign']))

In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm_lr, interpolation='nearest', cmap=plt.cm.Blues)
ax.set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'],
       title='Confusion Matrix - Logistic Regression',
       ylabel='True label', xlabel='Predicted label')
plt.colorbar(im)

for i in range(2):
    for j in range(2):
        ax.text(j, i, format(cm_lr[i, j], 'd'), ha='center', va='center',
                color='white' if cm_lr[i, j] > cm_lr.max()/2 else 'black')

plt.tight_layout()
plt.savefig('./data/confusion_matrix_logistic_regression.png', dpi=150, bbox_inches='tight')
plt.show()

# 4. Arquitetura da Rede Neural

In [ ]:
class CancerClassifier(nn.Module):
    def __init__(self, input_size, hidden_sizes=[64, 32], dropout_rate=0.2):
        super(CancerClassifier, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_size),
                nn.Dropout(dropout_rate)
            ])
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, 1))
        layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# 5. Função de Treinamento

In [ ]:
def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=100):
    train_losses = []
    test_losses = []
    train_accuracies = []
    test_accuracies = []
    
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            predicted = (outputs > 0.5).float()
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)
        
        train_loss /= len(train_loader.dataset)
        train_accuracy = correct_train / total_train
        
        model.eval()
        test_loss = 0.0
        correct_test = 0
        total_test = 0
        
        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs).squeeze()
                loss = criterion(outputs, labels)
                
                test_loss += loss.item() * inputs.size(0)
                predicted = (outputs > 0.5).float()
                correct_test += (predicted == labels).sum().item()
                total_test += labels.size(0)
        
        test_loss /= len(test_loader.dataset)
        test_accuracy = correct_test / total_test
        
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accuracies.append(train_accuracy)
        test_accuracies.append(test_accuracy)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}] '
                  f'Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f} '
                  f'Test Loss: {test_loss:.4f}, Test Acc: {test_accuracy:.4f}')
    
    return train_losses, test_losses, train_accuracies, test_accuracies

# 6. Treinar Modelo com Arquitetura [64, 32]

In [ ]:
input_size = X_train_scaled.shape[1]
model_64_32 = CancerClassifier(input_size, hidden_sizes=[64, 32], dropout_rate=0.2)

criterion = nn.BCELoss()
optimizer = optim.Adam(model_64_32.parameters(), lr=0.001)

print('Training model with architecture [64, 32]...')
train_losses_1, test_losses_1, train_acc_1, test_acc_1 = train_model(
    model_64_32, train_loader, test_loader, criterion, optimizer, num_epochs=100
)

# 7. Treinar Modelo com Arquitetura [128, 64, 32]

In [ ]:
model_128_64_32 = CancerClassifier(input_size, hidden_sizes=[128, 64, 32], dropout_rate=0.2)

criterion = nn.BCELoss()
optimizer = optim.Adam(model_128_64_32.parameters(), lr=0.001)

print('Training model with architecture [128, 64, 32]...')
train_losses_2, test_losses_2, train_acc_2, test_acc_2 = train_model(
    model_128_64_32, train_loader, test_loader, criterion, optimizer, num_epochs=100
)

# 8. Visualização do Progresso do Treinamento

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(train_losses_1, label='Train Loss', alpha=0.7)
axes[0, 0].plot(test_losses_1, label='Test Loss', alpha=0.7)
axes[0, 0].set_title('Model [64, 32] - Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(train_acc_1, label='Train Accuracy', alpha=0.7)
axes[0, 1].plot(test_acc_1, label='Test Accuracy', alpha=0.7)
axes[0, 1].set_title('Model [64, 32] - Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(train_losses_2, label='Train Loss', alpha=0.7)
axes[1, 0].plot(test_losses_2, label='Test Loss', alpha=0.7)
axes[1, 0].set_title('Model [128, 64, 32] - Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(train_acc_2, label='Train Accuracy', alpha=0.7)
axes[1, 1].plot(test_acc_2, label='Test Accuracy', alpha=0.7)
axes[1, 1].set_title('Model [128, 64, 32] - Accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Training Progress Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('./data/training_progress_nn.png', dpi=150, bbox_inches='tight')
plt.show()

# 9. Avaliação do Modelo

In [ ]:
def evaluate_model(model, test_loader, model_name):
    model.eval()
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs).squeeze()
            predicted = (outputs > 0.5).float()
            all_predictions.extend(predicted.numpy())
            all_labels.extend(labels.numpy())
    
    all_predictions = np.array(all_predictions)
    all_labels = np.array(all_labels)
    
    accuracy = accuracy_score(all_labels, all_predictions)
    
    print(f'{model_name}')
    print(f'Accuracy: {accuracy:.4f}')
    print(f'\nClassification Report:')
    print(classification_report(all_labels, all_predictions, target_names=['Malignant', 'Benign']))
    
    return accuracy, all_predictions

In [ ]:
accuracy_nn1, y_pred_nn1 = evaluate_model(model_64_32, test_loader, 'Neural Network [64, 32]')
accuracy_nn2, y_pred_nn2 = evaluate_model(model_128_64_32, test_loader, 'Neural Network [128, 64, 32]')

In [ ]:
cm_nn1 = confusion_matrix(y_test, y_pred_nn1)
cm_nn2 = confusion_matrix(y_test, y_pred_nn2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(cm_nn1, interpolation='nearest', cmap=plt.cm.Blues)
axes[0].set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'],
       title='Confusion Matrix - NN [64, 32]',
       ylabel='True label', xlabel='Predicted label')
plt.colorbar(im1, ax=axes[0])

for i in range(2):
    for j in range(2):
        axes[0].text(j, i, format(cm_nn1[i, j], 'd'), ha='center', va='center',
                color='white' if cm_nn1[i, j] > cm_nn1.max()/2 else 'black')

im2 = axes[1].imshow(cm_nn2, interpolation='nearest', cmap=plt.cm.Blues)
axes[1].set(xticks=[0, 1], yticks=[0, 1],
       xticklabels=['Malignant', 'Benign'], yticklabels=['Malignant', 'Benign'],
       title='Confusion Matrix - NN [128, 64, 32]',
       ylabel='True label', xlabel='Predicted label')
plt.colorbar(im2, ax=axes[1])

for i in range(2):
    for j in range(2):
        axes[1].text(j, i, format(cm_nn2[i, j], 'd'), ha='center', va='center',
                color='white' if cm_nn2[i, j] > cm_nn2.max()/2 else 'black')

plt.tight_layout()
plt.savefig('./data/confusion_matrix_nn.png', dpi=150, bbox_inches='tight')
plt.show()

# 10. Visualização da Fronteira de Decisão

Since the dataset has 30 features, we use PCA to reduce it to 2 dimensions for visualization purposes.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f'Explained variance ratio: {pca.explained_variance_ratio_}')
print(f'Total explained variance: {sum(pca.explained_variance_ratio_):.4f}')

In [ ]:
lr_pca = LogisticRegression(max_iter=10000, random_state=42)
lr_pca.fit(X_train_pca, y_train)

X_train_pca_tensor = torch.FloatTensor(X_train_pca)
y_train_pca_tensor = torch.FloatTensor(y_train)
X_test_pca_tensor = torch.FloatTensor(X_test_pca)
y_test_pca_tensor = torch.FloatTensor(y_test)

train_dataset_pca = TensorDataset(X_train_pca_tensor, y_train_pca_tensor)
test_dataset_pca = TensorDataset(X_test_pca_tensor, y_test_pca_tensor)

train_loader_pca = DataLoader(train_dataset_pca, batch_size=32, shuffle=True)
test_loader_pca = DataLoader(test_dataset_pca, batch_size=32, shuffle=False)

model_pca_64_32 = CancerClassifier(2, hidden_sizes=[64, 32], dropout_rate=0.2)
criterion_pca = nn.BCELoss()
optimizer_pca = optim.Adam(model_pca_64_32.parameters(), lr=0.001)

train_model(model_pca_64_32, train_loader_pca, test_loader_pca, criterion_pca, optimizer_pca, num_epochs=100)

In [ ]:
def plot_decision_boundary(model, X, y, title, ax, is_torch=True):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    if is_torch:
        model.eval()
        with torch.no_grad():
            Z = model(torch.FloatTensor(grid_points))
            Z = (Z.numpy() > 0.5).astype(int)
    else:
        Z = model.predict(grid_points)
    
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=plt.cm.RdBu)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='red', label='Malignant', edgecolors='k', s=50)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='blue', label='Benign', edgecolors='k', s=50)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(title)
    ax.legend()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

plot_decision_boundary(lr_pca, X_test_pca, y_test, 'Logistic Regression', axes[0], is_torch=False)
plot_decision_boundary(model_pca_64_32, X_test_pca, y_test, 'NN [64, 32]', axes[1], is_torch=True)

plt.suptitle('Decision Boundaries (PCA-reduced to 2D)', fontsize=14)
plt.tight_layout()
plt.savefig('./data/decision_boundaries_nn.png', dpi=150, bbox_inches='tight')
plt.show()

# 11. Comparação: Regressão Logística vs Redes Neurais

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'NN [64, 32]', 'NN [128, 64, 32]'],
    'Accuracy': [accuracy_lr, accuracy_nn1, accuracy_nn2]
})

print('Model Comparison:')
print(comparison.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(comparison['Model'], comparison['Accuracy'], color=['steelblue', 'coral', 'green'])
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Comparison: Logistic Regression vs Neural Networks')
ax.set_ylim(0.8, 1.0)

for bar, acc in zip(bars, comparison['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('./data/accuracy_comparison_nn.png', dpi=150, bbox_inches='tight')
plt.show()

# 12. Análise dos Resultados

## Arquiteturas Testadas

Foram testadas duas arquiteturas de redes neurais:
1. **[64, 32]**: Duas camadas ocultas com 64 e 32 neurônios
2. **[128, 64, 32]**: Três camadas ocultas com 128, 64 e 32 neurônios

Ambas as arquiteturas utilizaram:
- Função de ativação ReLU
- Batch Normalization para estabilizar o treinamento
- Dropout (0.2) para regularização
- Função de perda BCELoss (Binary Cross Entropy)
- Otimizador Adam com learning rate de 0.001

## Comparação com Regressão Logística

A regressão logística já demonstrou um desempenho muito bom neste dataset (acurácia ~97%), o que era esperado dado que:
- O dataset é linearmente separável na maioria dos casos
- As features já foram computadas de forma a separar bem as classes

As redes neurais conseguiram atingir acurácia similar ou ligeiramente superior, mas a diferença não é significativa para este problema específico.

## Vantagens da Rede Neural

1. **Capacidade de aprendizado não-linear**: A rede neural pode capturar relações mais complexas entre as features
2. **Flexibilidade de arquitetura**: É possível ajustar o número de camadas e neurônios conforme a complexidade do problema
3. **Generalização**: Com dropout e batch normalization, a rede tende a generalizar melhor para novos dados

## Desvantagens da Rede Neural

1. **Complexidade**: Mais parâmetros para ajustar e mais hiperparâmetros para otimizar
2. **Tempo de treinamento**: Requer mais tempo de computação que a regressão logística
3. **Interpretabilidade**: Difícil entender quais features são mais importantes para a decisão

## Conclusão

Para este dataset específico, a regressão logística é uma escolha mais simples e eficiente. No entanto, as redes neurais demonstram ser uma alternativa viável e podem ser mais úteis para problemas mais complexos onde a relação entre features e target não é linear.

In [ ]:
import os
print('Files saved in data/ folder:')
for f in sorted(os.listdir('data')):
    if f.endswith('.png') or f.endswith('.csv'):
        print(f'  - {f}')